In [1]:
"""Stufe 3: Globales Gradient-Boosting (LightGBM), Bericht 3.8.
Ausführen: uv run python src/gbm.py   (Ausgabe: reports/gbm_output.txt, reports/results_gbm.csv, reports/gbm_importance.csv)
Vorher: src/clean.py, src/baselines.py und src/statistical.py müssen gelaufen sein.
"""

'Stufe 3: Globales Gradient-Boosting (LightGBM), Bericht 3.8.\nAusführen: uv run python src/gbm.py   (Ausgabe: reports/gbm_output.txt, reports/results_gbm.csv, reports/gbm_importance.csv)\nVorher: src/clean.py, src/baselines.py und src/statistical.py müssen gelaufen sein.\n'

In [2]:
import sys
import time
from pathlib import Path

import pandas as pd

ROOT = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd().resolve().parent  # Skript oder Notebook
sys.path.insert(0, str(ROOT / "src"))
from forecast_eval import WINDOWS, load_clean, score, seasonal_baselines, to_wide, window_frames  # noqa: E402
from gbm_model import Panel, fit_predict  # noqa: E402
from stat_models import ets_forecast  # noqa: E402

pd.set_option("display.width", 170)
lines = []


def out(msg=""):
    print(msg)
    lines.append(str(msg))


df = load_clean()
W_train = to_wide(df, "sales_train")
panel = Panel(df)
out(f"Reihen: {panel.S}, Tage: {panel.T}")

Reihen: 1782, Tage: 1688


## 1. Das Modell
**Ein** Modell für alle 1.782 Reihen (LightGBM, 500 Bäume). Eine Trainingszeile beschreibt eine Reihe, einen Zieltag und den
Abstand `h` (1 bis 16 Tage) zum Prognose-Ursprung. Zielgröße `log1p(Absatz)`.

**Eingaben aus der Vergangenheit** (nur Tage bis zum Ursprung):
Mittel der letzten 7, 28, 91 und 364 Tage, letzter bekannter Wert, Tage seit diesem Wert, Alter der Reihe, die 4 letzten
Werte des gleichen Wochentags wie der Zieltag und ihr Mittel.

**Eingaben, die auch in der Zukunft bekannt sind:** Wochentag, Tag im Monat, Monat, Feiertag, Ereignis, Arbeitstag-Ausgleich,
Aktion (`onpromotion`), `promo_known`, Filiale, Familie, Filialtyp, Stadt, Region, Cluster.

**Training:** je Fenster die 730 Tage vor dem Ursprung, je (Reihe, Zieltag) zwei zufällige Abstände. Das Label eines
Trainingstags liegt immer vor dem Ursprung. Maskierte Tage (`sales_train` = NaN) sind weder Label noch Eingabe.
Ohne Ölpreis (Bericht 3.4, Entscheidung 8).

In [3]:
rows, per_family_w1, importances, models = [], {}, [], {}
for name, origin in WINDOWS.items():
    t0 = time.time()
    pred, model, n_rows = fit_predict(panel, origin)
    truth, excluded, outlier = window_frames(df, origin)
    r = score(truth, pred, excluded, outlier)
    rows.append({"Fenster": name, "Modell": "LightGBM", "RMSLE": r["rmsle"], "RMSLE ohne Spitzen": r["rmsle_ohne_spitzen"],
                 "bewertet": r["n_bewertet"], "ohne Prognose": r["n_ohne_prognose"]})
    if name.startswith("W1"):
        per_family_w1 = r["je_familie"]
    importances.append(pd.Series(model.feature_importance("gain"), index=model.feature_name(), name=name))
    out(f"{name}: RMSLE {r['rmsle']:.4f} (Trainingszeilen {n_rows:,}, {time.time() - t0:.0f} s)")
res = pd.DataFrame(rows)

W1 Aug 2017: RMSLE 0.3939 (Trainingszeilen 2,482,918, 43 s)


W2 Dez 2016: RMSLE 0.4358 (Trainingszeilen 2,393,324, 39 s)


W3 Mär 2017: RMSLE 0.3987 (Trainingszeilen 2,428,856, 40 s)


## 2. Vergleich mit Benchmark und statistischem Modell
Erfolgskriterium (Bericht 3.4): RMSLE mindestens 10 % unter der besten Baseline, im Mittel der drei Fenster
und in mindestens zwei Fenstern einzeln.

In [4]:
base = pd.read_csv(ROOT / "reports" / "results_baselines.csv")
bench = base.pivot(index="Fenster", columns="Modell", values="RMSLE").min(axis=1).reindex(list(WINDOWS))
ets = pd.read_csv(ROOT / "reports" / "results_statistical.csv").set_index("Fenster").RMSLE.reindex(list(WINDOWS))
gbm = res.set_index("Fenster").RMSLE
cmp_ = pd.DataFrame({"Benchmark": bench, "ETS": ets, "LightGBM": gbm})
cmp_["LightGBM vs. Benchmark"] = (cmp_.LightGBM / cmp_.Benchmark - 1).map("{:+.1%}".format)
cmp_["LightGBM vs. ETS"] = (cmp_.LightGBM / cmp_.ETS - 1).map("{:+.1%}".format)
cmp_["Ziel (-10 %)"] = (0.9 * cmp_.Benchmark)
cmp_["erreicht"] = cmp_.LightGBM <= cmp_["Ziel (-10 %)"]
out("\n" + cmp_.round(4).to_string())
out(f"\nMittel: Benchmark {bench.mean():.4f}, ETS {ets.mean():.4f}, LightGBM {gbm.mean():.4f} "
    f"({gbm.mean() / bench.mean() - 1:+.1%} gegen Benchmark; Ziel {0.9 * bench.mean():.4f})")
out(f"Erfolgskriterium: Mittel erreicht: {gbm.mean() <= 0.9 * bench.mean()}; Fenster einzeln erreicht: {int(cmp_['erreicht'].sum())} von 3")


             Benchmark     ETS  LightGBM LightGBM vs. Benchmark LightGBM vs. ETS  Ziel (-10 %)  erreicht
Fenster                                                                                                 
W1 Aug 2017     0.5306  0.4845    0.3939                 -25.8%           -18.7%        0.4775      True
W2 Dez 2016     0.5938  0.5512    0.4358                 -26.6%           -20.9%        0.5345      True
W3 Mär 2017     0.4523  0.4108    0.3987                 -11.8%            -2.9%        0.4071      True

Mittel: Benchmark 0.5256, ETS 0.4822, LightGBM 0.4095 (-22.1% gegen Benchmark; Ziel 0.4730)
Erfolgskriterium: Mittel erreicht: True; Fenster einzeln erreicht: 3 von 3


## 3. Wo gewinnt LightGBM gegenüber ETS? Fehler je Familie, Fenster W1

In [5]:
o1 = WINDOWS["W1 Aug 2017"]
truth, excluded, outlier = window_frames(df, o1)
b2 = seasonal_baselines(W_train.loc[:o1], o1)["B2 Mittel 4 Wochen"]
ets_w1, _ = ets_forecast(W_train, o1, fallback=b2)
ets_fam = score(truth, ets_w1, excluded, outlier)["je_familie"]
fam = pd.DataFrame({"ETS": ets_fam, "LightGBM": per_family_w1})
fam["Änderung"] = fam.LightGBM / fam.ETS - 1
out("\nFamilien in W1, größte Verbesserung gegenüber ETS zuerst:\n" + fam.sort_values("Änderung").assign(
    Änderung=lambda d: d.Änderung.map("{:+.1%}".format)).round(3).to_string())
out(f"\nFamilien mit besserem LightGBM: {(fam.LightGBM < fam.ETS).sum()} von {len(fam)}")


Familien in W1, größte Verbesserung gegenüber ETS zuerst:
                              ETS  LightGBM Änderung
SCHOOL AND OFFICE SUPPLIES  1.535     0.682   -55.6%
CLEANING                    0.424     0.296   -30.1%
PERSONAL CARE               0.282     0.199   -29.5%
GROCERY I                   0.198     0.151   -23.7%
DAIRY                       0.175     0.143   -18.3%
HOME CARE                   0.292     0.244   -16.5%
LIQUOR,WINE,BEER            0.517     0.435   -15.9%
BEAUTY                      0.554     0.470   -15.3%
FROZEN FOODS                0.305     0.265   -13.1%
PRODUCE                     0.157     0.139   -11.6%
BREAD/BAKERY                0.191     0.169   -11.4%
LINGERIE                    0.693     0.617   -11.0%
DELI                        0.204     0.182   -10.9%
GROCERY II                  0.712     0.637   -10.4%
BEVERAGES                   0.250     0.226    -9.5%
POULTRY                     0.222     0.201    -9.3%
MEATS                       0.222     0.

## 4. Was nutzt das Modell? Wichtigkeit der Merkmale
Die Wichtigkeit (Gain) sagt, wie viel Fehler ein Merkmal beim Trainieren insgesamt abgebaut hat. Sie zeigt
**Nutzung**, nicht Ursache: Ein wichtiges Merkmal ist nicht automatisch der Grund für den Absatz.

In [6]:
imp = pd.concat(importances, axis=1)
imp["Anteil"] = (imp.mean(axis=1) / imp.mean(axis=1).sum())
imp = imp.sort_values("Anteil", ascending=False)
out("\nMerkmalswichtigkeit (Anteil am Gesamt-Gain, Mittel der drei Fenster):\n" + imp[["Anteil"]].assign(
    Anteil=lambda d: (d.Anteil * 100).round(1).astype(str) + " %").to_string())


Merkmalswichtigkeit (Anteil am Gesamt-Gain, Mittel der drei Fenster):
                           Anteil
gleicher_wochentag_mittel  58.0 %
mittel_7d                  27.2 %
gleicher_wochentag_1        8.7 %
letzter_wert                1.8 %
mittel_28d                  1.4 %
mittel_91d                  0.7 %
family                      0.5 %
mittel_364d                 0.4 %
aktion                      0.2 %
monat                       0.2 %
alter_reihe                 0.2 %
store                       0.2 %
tag_im_monat                0.1 %
gleicher_wochentag_4        0.1 %
wochentag                   0.1 %
gleicher_wochentag_2        0.1 %
gleicher_wochentag_3        0.1 %
feiertag                    0.0 %
city                        0.0 %
h                           0.0 %
type                        0.0 %
ereignis                    0.0 %
cluster                     0.0 %
tage_seit_letztem_wert      0.0 %
state                       0.0 %
arbeitstag_ausgleich        0.0 %
aktion_beka

In [7]:
res.to_csv(ROOT / "reports" / "results_gbm.csv", index=False)
imp.to_csv(ROOT / "reports" / "gbm_importance.csv")
(ROOT / "reports" / "gbm_output.txt").write_text("\n".join(lines), encoding="utf-8")

3825